# 04 — Avaliação do potencial de impacto na locomoção

Este notebook implementa o **módulo de avaliação de risco** do projeto.

O modelo YOLOv8n já treinado detecta os objetos. A partir das detecções,
este notebook utiliza regras simples e interpretáveis para avaliar
o quanto cada objeto **pode interferir na locomoção**.

## Fluxo

```text
Imagem
   ↓
YOLOv8n
   ↓
objeto + bounding box + confiança
   ↓
classe do objeto
   +
posição horizontal
   +
ocupação da imagem
   ↓
pontuação de risco
   ↓
BAIXO / MÉDIO / ALTO
```

> Importante: a ocupação da imagem é usada apenas como uma **aproximação visual de proximidade**.
> Ela não representa distância real em metros.

## 1. Antes de executar

Baixe o arquivo `best.pt` que foi salvo no Google Drive durante o treinamento.

Coloque-o nesta pasta do projeto:

```text
tcc_visao_computacional/
└── models/
    └── yolov8n_baseline/
        └── best.pt
```

O dataset processado criado no notebook 02 deverá continuar em:

```text
data/processed/homeobjects_yolo/
```

## 2. Importações e localização da raiz do projeto

In [ ]:
from pathlib import Path
import random
import sys

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image, ImageDraw, ImageFont


def localizar_raiz_projeto() -> Path:
    atual = Path.cwd().resolve()

    for candidata in [atual, *atual.parents]:
        if (
            (candidata / "requirements.txt").exists()
            and (candidata / "src").exists()
        ):
            return candidata

    raise FileNotFoundError(
        "Não foi possível localizar a raiz do projeto."
    )


RAIZ_PROJETO = localizar_raiz_projeto()

if str(RAIZ_PROJETO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROJETO))

print("Raiz do projeto:", RAIZ_PROJETO)

## 3. Instalar/importar Ultralytics

Se a biblioteca já estiver instalada no ambiente virtual, esta célula apenas fará a importação.

In [ ]:
try:
    from ultralytics import YOLO
except ImportError:
    import subprocess

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "ultralytics"]
    )

    from ultralytics import YOLO

print("Ultralytics importado com sucesso.")

## 4. Caminhos do projeto

In [ ]:
CAMINHO_MODELO = (
    RAIZ_PROJETO
    / "models"
    / "yolov8n_baseline"
    / "best.pt"
)

PASTA_TEST = (
    RAIZ_PROJETO
    / "data"
    / "processed"
    / "homeobjects_yolo"
    / "images"
    / "test"
)

PASTA_RESULTADOS = (
    RAIZ_PROJETO
    / "reports"
    / "predictions"
    / "risco"
)

PASTA_TABELAS = RAIZ_PROJETO / "reports" / "tables"

PASTA_RESULTADOS.mkdir(parents=True, exist_ok=True)
PASTA_TABELAS.mkdir(parents=True, exist_ok=True)

if not CAMINHO_MODELO.exists():
    raise FileNotFoundError(
        f"best.pt não encontrado em:\n{CAMINHO_MODELO}\n\n"
        "Baixe o best.pt do Google Drive e coloque nessa pasta."
    )

if not PASTA_TEST.exists():
    raise FileNotFoundError(
        f"Pasta de teste não encontrada em:\n{PASTA_TEST}"
    )

print("Modelo:", CAMINHO_MODELO)
print("Imagens de teste:", PASTA_TEST)
print("Resultados:", PASTA_RESULTADOS)

# 5. Definição das regras de risco

Vamos separar as 12 classes em três grupos.

### Objetos avaliados como possíveis obstáculos físicos

- `bed`
- `sofa`
- `chair`
- `table`
- `wardrobe`
- `potted plant`

### Objetos contextuais

- `lamp`
- `door`

Eles podem ou não representar um obstáculo dependendo da situação.
O dataset não informa, por exemplo, se uma porta está aberta ou fechada
ou se uma luminária está no chão ou sobre uma mesa.

### Objetos ignorados na avaliação de locomoção

- `tv`
- `laptop`
- `window`
- `photo frame`

O YOLO continua detectando essas classes, mas elas não recebem
classificação automática de risco nesta primeira versão.

In [ ]:
PESO_OBJETO = {
    "bed": 3,
    "sofa": 3,
    "chair": 3,
    "table": 3,
    "wardrobe": 3,
    "potted plant": 2,
}

CLASSES_CONTEXTUAIS = {
    "lamp",
    "door",
}

CLASSES_IGNORADAS = {
    "tv",
    "laptop",
    "window",
    "photo frame",
}

print("Possíveis obstáculos:", sorted(PESO_OBJETO))
print("Contextuais:", sorted(CLASSES_CONTEXTUAIS))
print("Ignoradas:", sorted(CLASSES_IGNORADAS))

## 6. Regras espaciais

A imagem será dividida horizontalmente em três regiões:

```text
┌────────────┬────────────┬────────────┐
│            │            │            │
│  esquerda  │   centro   │   direita  │
│            │            │            │
└────────────┴────────────┴────────────┘
```

O centro horizontal da bounding box define a posição.

Para a ocupação da imagem, utilizamos:

```text
área da bounding box / área total da imagem
```

Nesta primeira versão serão utilizados limites heurísticos:

- pequena: menos de 5%;
- média: entre 5% e 20%;
- grande: 20% ou mais.

Esses limites são **parâmetros iniciais** e deverão ser avaliados posteriormente.

In [ ]:
LIMITE_ESQUERDA = 1 / 3
LIMITE_DIREITA = 2 / 3

LIMITE_AREA_MEDIA = 0.05
LIMITE_AREA_GRANDE = 0.20

PESO_POSICAO = {
    "esquerda": 1,
    "centro": 3,
    "direita": 1,
}

PESO_TAMANHO = {
    "pequeno": 1,
    "medio": 2,
    "grande": 3,
}

## 7. Funções de avaliação

A pontuação será calculada somente para classes tratadas como possíveis obstáculos:

```text
pontuação =
    peso do objeto
    + peso da posição
    + peso do tamanho
```

Classificação inicial:

```text
3–4  → baixo
5–6  → médio
7–9  → alto
```

As classes contextuais recebem o status `contextual`.
As classes sem relevância para locomoção recebem `ignorado`.

In [ ]:
def classificar_posicao(
    centro_x: float,
    largura_imagem: int,
) -> str:
    centro_normalizado = centro_x / largura_imagem

    if centro_normalizado < LIMITE_ESQUERDA:
        return "esquerda"

    if centro_normalizado < LIMITE_DIREITA:
        return "centro"

    return "direita"


def classificar_tamanho(
    area_box: float,
    area_imagem: float,
):
    proporcao = area_box / area_imagem

    if proporcao < LIMITE_AREA_MEDIA:
        return "pequeno", proporcao

    if proporcao < LIMITE_AREA_GRANDE:
        return "medio", proporcao

    return "grande", proporcao


def classificar_risco(
    classe: str,
    posicao: str,
    tamanho: str,
):
    if classe in CLASSES_IGNORADAS:
        return {
            "pontuacao": 0,
            "nivel_risco": "ignorado",
            "motivo": "classe sem relevância direta para locomoção",
        }

    if classe in CLASSES_CONTEXTUAIS:
        return {
            "pontuacao": None,
            "nivel_risco": "contextual",
            "motivo": "a classe exige informação adicional de contexto",
        }

    if classe not in PESO_OBJETO:
        return {
            "pontuacao": None,
            "nivel_risco": "nao_mapeado",
            "motivo": "classe não configurada",
        }

    pontuacao = (
        PESO_OBJETO[classe]
        + PESO_POSICAO[posicao]
        + PESO_TAMANHO[tamanho]
    )

    if pontuacao <= 4:
        nivel = "baixo"
    elif pontuacao <= 6:
        nivel = "medio"
    else:
        nivel = "alto"

    return {
        "pontuacao": pontuacao,
        "nivel_risco": nivel,
        "motivo": (
            f"{classe} + posição {posicao} + tamanho {tamanho}"
        ),
    }

## 8. Carregar o modelo treinado

In [ ]:
modelo = YOLO(str(CAMINHO_MODELO))

print("Modelo carregado.")
print("Classes:")
print(modelo.names)

# 9. Função principal de análise de uma imagem

Esta função:

1. executa o YOLO;
2. extrai cada bounding box;
3. calcula posição;
4. calcula ocupação da imagem;
5. atribui a classificação de risco;
6. retorna uma tabela com todos os objetos detectados.

In [ ]:
def analisar_imagem(
    caminho_imagem,
    confianca_minima=0.25,
):
    caminho_imagem = Path(caminho_imagem)

    imagem = Image.open(caminho_imagem).convert("RGB")
    largura, altura = imagem.size
    area_imagem = largura * altura

    resultados = modelo.predict(
        source=str(caminho_imagem),
        conf=confianca_minima,
        verbose=False,
    )

    resultado = resultados[0]
    registros = []

    if resultado.boxes is None:
        return pd.DataFrame(), imagem

    for box in resultado.boxes:
        classe_id = int(box.cls[0].item())
        classe = modelo.names[classe_id]
        confianca = float(box.conf[0].item())

        x1, y1, x2, y2 = [
            float(valor)
            for valor in box.xyxy[0].tolist()
        ]

        largura_box = x2 - x1
        altura_box = y2 - y1
        area_box = largura_box * altura_box

        centro_x = (x1 + x2) / 2

        posicao = classificar_posicao(
            centro_x=centro_x,
            largura_imagem=largura,
        )

        tamanho, proporcao_area = classificar_tamanho(
            area_box=area_box,
            area_imagem=area_imagem,
        )

        avaliacao = classificar_risco(
            classe=classe,
            posicao=posicao,
            tamanho=tamanho,
        )

        registros.append(
            {
                "arquivo": caminho_imagem.name,
                "classe": classe,
                "confianca": confianca,
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
                "posicao": posicao,
                "ocupacao_imagem": proporcao_area,
                "tamanho": tamanho,
                "pontuacao": avaliacao["pontuacao"],
                "nivel_risco": avaliacao["nivel_risco"],
                "motivo": avaliacao["motivo"],
            }
        )

    return pd.DataFrame(registros), imagem

## 10. Selecionar uma imagem de teste

Inicialmente utilizaremos uma imagem aleatória do conjunto TEST.

A semente fixa garante que o mesmo exemplo seja selecionado ao executar novamente o notebook.

In [ ]:
EXTENSOES_IMAGEM = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

imagens_test = sorted(
    arquivo
    for arquivo in PASTA_TEST.rglob("*")
    if arquivo.is_file()
    and arquivo.suffix.lower() in EXTENSOES_IMAGEM
)

if not imagens_test:
    raise RuntimeError("Nenhuma imagem encontrada no conjunto TEST.")

random.seed(42)
imagem_teste = random.choice(imagens_test)

print("Imagem selecionada:")
print(imagem_teste)

## 11. Executar a avaliação

In [ ]:
df_deteccoes, imagem_original = analisar_imagem(
    imagem_teste,
    confianca_minima=0.25,
)

colunas_exibicao = [
    "classe",
    "confianca",
    "posicao",
    "ocupacao_imagem",
    "tamanho",
    "pontuacao",
    "nivel_risco",
]

if df_deteccoes.empty:
    print("Nenhum objeto detectado.")
else:
    display(
        df_deteccoes[colunas_exibicao]
        .sort_values(
            ["nivel_risco", "confianca"],
            ascending=[True, False],
        )
    )

# 12. Visualizar as detecções e o nível de risco

In [ ]:
def desenhar_resultados(
    imagem,
    df,
):
    imagem_saida = imagem.copy()
    desenho = ImageDraw.Draw(imagem_saida)

    largura, altura = imagem_saida.size

    # Linhas que representam as três regiões horizontais.
    x_esquerda = int(largura * LIMITE_ESQUERDA)
    x_direita = int(largura * LIMITE_DIREITA)

    desenho.line(
        [(x_esquerda, 0), (x_esquerda, altura)],
        width=2,
    )
    desenho.line(
        [(x_direita, 0), (x_direita, altura)],
        width=2,
    )

    for _, linha in df.iterrows():
        caixa = (
            linha["x1"],
            linha["y1"],
            linha["x2"],
            linha["y2"],
        )

        desenho.rectangle(
            caixa,
            width=3,
        )

        texto = (
            f'{linha["classe"]} | '
            f'{linha["nivel_risco"]} | '
            f'{linha["confianca"]:.2f}'
        )

        desenho.text(
            (linha["x1"], max(linha["y1"] - 15, 0)),
            texto,
        )

    return imagem_saida


imagem_anotada = desenhar_resultados(
    imagem_original,
    df_deteccoes,
)

plt.figure(figsize=(14, 9))
plt.imshow(imagem_anotada)
plt.axis("off")
plt.title("Detecções + avaliação de impacto na locomoção")
plt.show()

## 13. Gerar uma mensagem simplificada

Nesta etapa ainda não usamos áudio.

Primeiro vamos validar se a lógica gera mensagens úteis.

In [ ]:
def gerar_mensagens(df):
    mensagens = []

    if df.empty:
        return ["Nenhum objeto relevante detectado."]

    riscos = df[
        df["nivel_risco"].isin(["medio", "alto"])
    ].copy()

    ordem = {
        "alto": 0,
        "medio": 1,
    }

    if riscos.empty:
        return ["Nenhum obstáculo relevante identificado."]

    riscos["ordem"] = riscos["nivel_risco"].map(ordem)

    riscos = riscos.sort_values(
        ["ordem", "ocupacao_imagem"],
        ascending=[True, False],
    )

    for _, linha in riscos.iterrows():
        if linha["nivel_risco"] == "alto":
            prefixo = "Cuidado"
        else:
            prefixo = "Atenção"

        mensagens.append(
            f'{prefixo}: {linha["classe"]} '
            f'{linha["posicao"]}.'
        )

    return mensagens


mensagens = gerar_mensagens(df_deteccoes)

for mensagem in mensagens:
    print(mensagem)

# 14. Testar várias imagens

Uma única imagem não é suficiente para validar a regra.

Vamos selecionar **20 imagens aleatórias** do conjunto TEST e registrar todas as detecções e avaliações.

In [ ]:
random.seed(42)

quantidade_amostras = min(20, len(imagens_test))
amostras = random.sample(
    imagens_test,
    quantidade_amostras,
)

resultados_amostras = []

for indice, caminho_imagem in enumerate(amostras, start=1):
    print(
        f"[{indice:02d}/{quantidade_amostras}] "
        f"{caminho_imagem.name}"
    )

    df_imagem, _ = analisar_imagem(
        caminho_imagem,
        confianca_minima=0.25,
    )

    if not df_imagem.empty:
        resultados_amostras.append(df_imagem)

if resultados_amostras:
    df_amostras = pd.concat(
        resultados_amostras,
        ignore_index=True,
    )
else:
    df_amostras = pd.DataFrame()

print()
print("Detecções analisadas:", len(df_amostras))

## 15. Resumo dos níveis produzidos

Essa tabela não mede ainda se a classificação de risco está correta.

Ela serve para entendermos **como nossa regra está se comportando**.

In [ ]:
if not df_amostras.empty:
    resumo_risco = (
        df_amostras
        .groupby(
            ["nivel_risco", "classe"],
            dropna=False,
        )
        .size()
        .reset_index(name="quantidade")
        .sort_values(
            ["nivel_risco", "quantidade"],
            ascending=[True, False],
        )
    )

    display(resumo_risco)

## 16. Salvar resultados da primeira avaliação

In [ ]:
if not df_amostras.empty:
    caminho_csv = (
        PASTA_TABELAS
        / "avaliacao_risco_amostras.csv"
    )

    df_amostras.to_csv(
        caminho_csv,
        index=False,
        encoding="utf-8-sig",
    )

    print("Arquivo salvo em:")
    print(caminho_csv)

caminho_imagem_saida = (
    PASTA_RESULTADOS
    / "exemplo_avaliacao_risco.jpg"
)

imagem_anotada.save(caminho_imagem_saida)

print()
print("Imagem salva em:")
print(caminho_imagem_saida)

# 17. O que este notebook já resolve

Neste ponto temos dois módulos claramente separados:

```text
MÓDULO 1
YOLOv8n
    ↓
reconhecimento dos objetos

MÓDULO 2
regras de avaliação
    ↓
classe
+ posição
+ ocupação da imagem
    ↓
potencial de impacto na locomoção
```

Isso é importante porque a classificação de risco fica **interpretável**.

Por exemplo:

```text
chair
+ centro
+ grande
↓
pontuação alta
↓
risco alto
```

## Limitação atual

A regra ainda é uma **heurística inicial**.

Ela ainda precisa ser validada com cenários preparados especificamente para verificar:

- se o objeto realmente bloqueia a passagem;
- se a posição atribuída faz sentido;
- se os limites de tamanho são adequados;
- se as faixas baixo/médio/alto representam corretamente o impacto na locomoção.

Essa validação será a próxima etapa do projeto.

# Próxima etapa

Depois de executar este notebook e analisar visualmente alguns exemplos,
vamos construir uma pequena **base de cenários de validação**.

Exemplo:

| Objeto | Posição | Tamanho | Resultado esperado |
|---|---|---|---|
| chair | centro | grande | alto |
| chair | esquerda | pequeno | baixo/médio |
| table | centro | grande | alto |
| potted plant | direita | pequeno | baixo |
| photo frame | centro | grande | ignorado |

A partir disso poderemos medir se a **regra de risco** está produzindo resultados coerentes e ajustar os parâmetros antes de adicionar áudio.